# 39 — the connector arm: A2's run with LoRA reaching the ViT→LLM merger

Rung 39. Baseline = rung 21 arm `A2_lr`, run `21_lr_2e4_v1`, **epoch 1**, `checkpoint-901`.
The ONE variable: `--target_modules` gains the eight merger `Linear` layers, as **`all-linear` PLUS
the eight names**. Everything else is A2, verbatim.

**Pre-registration: [`PLAN.md`](PLAN.md).** It was committed before this notebook existed.

- **Declared primary cell (RULES §S3): `object_recognition_ID`, control `0.6087962962962963`.** The
  connector carries visual features into the LLM, so `object_recognition` is the bucket most directly
  downstream of the intervention. It is **not** local `bucket_mean`, which overstates the judge by
  +0.12 and inverts the bucket ordering.
- **A WIN needs** (a) the paired video-clustered CI on that cell to exclude zero in the arm's favour,
  (b) that to hold on ID and OOD jointly, and (c) **no** cell anywhere showing significant harm.
  Any cell may veto; only the declared cell may grant. A positive point estimate whose CI includes
  zero is a **NULL** (RULES §S8).
- **Schedule (A1, `PLAN.md` §6a):** `--num_train_epochs 3`, trainer TERMINATED once `checkpoint-901`
  is complete. Shortening the schedule would move warmup 81 → 27 and put LR at 0.0 instead of
  mid-cosine — a second, undeclared variable.
- **A faithful negative is a real result** and is recorded as one.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import glob, json, logging, os, shutil, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT inherit the
# env's bin/ on PATH, and it must be THIS interpreter's bin.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models", EXP / "_tools",
          REPO / "experiments" / "21-recipe-sweep" / "_models",
          REPO / "experiments" / "18-count-aug" / "_models",
          REPO / "experiments" / "06-vit-lora" / "_models",
          REPO / "experiments" / "02-lora-sft" / "_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

# 🔴 HF_HOME must be set BEFORE the offline flags mean anything, and before any HF import. The SDK's
# judge is cached at /workspace/hf_cache, NOT at the default ~/.cache/huggingface — which is empty on
# this pod. With OFFLINE set and HF_HOME unset, transformers raises from inside `run_baseline`, i.e.
# AFTER the 17 GB merge and the whole inference pass.
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.run import run_baseline
import connector_lora_train as engine
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "| repo:", REPO)


## Sibling rungs (change ONE value, rerun, bump the tag)

- **39a:** `STOP_AT_STEP = 1802` — read epoch 2 against A2's own epoch 2 (`checkpoint-1802`,
  `proxy_leaderboard` 0.5751). Epoch-matched by construction, RULES §6b.
- **39b:** `STOP_AT_STEP = None` — run all three epochs and read the whole curve. ~8.7 h of GPU plus
  two more merge+eval cycles; the pod bill roughly triples.

🔴 **NOT a sibling rung:** `--num_train_epochs 1`. That rebuilds the cosine over 901 planned steps —
warmup 27 instead of 81, LR 0.0 at step 901 instead of mid-descent — and is a second undeclared
variable, not a cheaper version of this run (`PLAN.md` §6a).


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ----------
SMOKE        = True     # True -> 40 questions, wiring only. Full: -p SMOKE False
RUN          = "39_connector_v1"
REPO_ROOT    = "/workspace/repo_rodri"
DATA_ROOT    = "/workspace/orena-data"
CONTROL_RUN  = "/workspace/repo_rodri/experiments/21-recipe-sweep/runs/21_lr_2e4_v1"
CONTROL_EPOCH_DIR = "ep1_full"
STOP_AT_STEP = 901      # A1: terminate once checkpoint-901 is complete. None -> all 3 epochs.
KEEP_MERGED  = False    # a merged checkpoint is ~17 GB; keep only the one we ship


In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
from dataclasses import replace

RUN_TAG = "ep1_smoke" if SMOKE else "ep1_full"

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING, and fail
# loudly: an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"

cfg = engine.ConnectorConfig(
    exp_dir=EXP, run_name=RUN, data_root=DATA_ROOT, stop_at_step=STOP_AT_STEP,
    control_train_jsonl=Path(REPO_ROOT) / "experiments" / "18-count-aug" / "runs"
                        / "18_count_aug_v1" / "train.jsonl",
)
RUN_DIR = cfg.run_dir
CTRL_DIR = Path(CONTROL_RUN) / CONTROL_EPOCH_DIR

# 🔑 `--freeze_aligner` is set from the GATE's own artifact, never from a remembered verdict — the
# rule rung 21 learned the hard way (read the artifact, never the declared variable). Both branches
# were pre-declared in PLAN.md §5 before the gate ran.
import reachability_gate as gate
gate_result = json.loads(Path(gate.RESULTS_JSON).read_text(encoding="utf-8"))
cfg = engine.apply_branch(cfg, gate_result)

print(f"SMOKE {SMOKE} | run {RUN} | tag {RUN_TAG}")
print(f"branch {gate_result['branch']} -> --freeze_aligner {str(cfg.freeze_aligner).lower()}")
print(f"stop_at_step {cfg.stop_at_step} (A1: the 3-epoch cosine, terminated at epoch 1)")
print(f"control {CTRL_DIR}   (rung 21 {engine.A2_RUN} arm {engine.A2_ARM} {engine.A2_CHECKPOINT})")


In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ---
# `run_baseline` loads the judge only AFTER the merge and the full inference pass, so a missing judge
# cache fails ~45 minutes in with everything already paid for. This is that failure, hoisted to the
# front and made cheap: the tokenizer alone proves the cache resolves under the offline flags.
# RAISES (RULES §7).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. The judge is cached at /workspace/hf_cache on this "
        "pod, not at the default ~/.cache/huggingface. Fix the env — do NOT disable the offline "
        "flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")


In [ ]:
# --- ALL GATES BEFORE THE GPU ---------------------------------------------------
# Every one of these RAISES, and every one of them exists because something once failed silently.
sv = engine.assert_single_variable_39(cfg)
print("single variable OK")
print("   branch         :", sv["branch"])
print("   declared flags :", sv["declared_flags"])
for flag, (ctrl_v, arm_v) in sv["diff"].items():
    print(f"   {flag}: control={ctrl_v} -> arm={arm_v}")
assert set(sv["diff"]) <= engine.ARMS["E_connector"], "a second variable reached the argv"

# the dataset is the CONTROL's bytes, not a re-export that happens to look the same
ds = engine.assert_dataset_is_the_controls(cfg)
print(f"dataset OK: {ds['n_rows']} rows, sha256 {ds['sha256'][:16]}…, "
      f"{ds['steps_per_epoch']:.0f} steps/epoch")
assert round(ds["steps_per_epoch"]) == engine.STEPS_PER_EPOCH, \
    f"{ds['steps_per_epoch']} steps/epoch — checkpoint-901 would not be epoch 1"

# 🔴 loss identically 0.0 is what an unsupervised corpus produces, and it exits rc=0 (rung 30)
print("supervised rows:", engine.assert_supervised(Path(cfg.train_jsonl)))

# WARNS, never blocks. `du` against the configured quota — df/statvfs report the MooseFS cluster
# at ~314 TB while the volume carries an invisible ~640 GB quota, and a run already died mid-merge.
print("disk:", engine.warn_disk())


In [ ]:
# --- train (A1: the control's 3-epoch cosine, terminated at checkpoint-901) ------
t0 = time.perf_counter()
train = engine.main(cfg, "train")
print(f"train stage finished in {(time.perf_counter() - t0) / 60:.1f} min")
print(json.dumps(train, indent=2))


In [ ]:
# --- POST-RUN: did it actually train, and did it actually reach the merger? -----
# 🔴 `rc=0` is NOT evidence. AdamW's decoupled weight decay moves every tensor at zero gradient, so a
# checkpoint diff cannot separate a real run from a no-op — only sum|Δ| does (1.34 vs 252.2
# measured). The instrument is the grad_norm log.
learned = engine.assert_learned(cfg)
print(f"learned OK: {learned['nonzero_grad_steps']}/{learned['logged_steps']} steps with gradient "
      f"({learned['nonzero_grad_frac']:.1%}), mean loss {learned['loss_mean']}")

# Two independent readings: ms-swift's own `lora_config:` line, and the tensor census of the REAL
# produced checkpoint. The gate's smoke result does not license the arm's result.
reached = engine.assert_merger_reached(cfg)
print(f"merger reached OK: n_aligner={reached['n_aligner']} n_llm={reached['n_llm']} "
      f"n_vit={reached['n_vit']} orphans={reached['n_orphans']} "
      f"(A2: 0 / {gate.A2_N_LLM} / {gate.A2_N_VIT} / 0)")
(EXP / f"RESULTS_gcov_arm_{RUN_TAG}.json").write_text(
    json.dumps({"train": train, "learned": learned, "reached": reached,
                "single_variable": {k: str(v) for k, v in sv.items()}}, indent=2, default=str),
    encoding="utf-8")


In [ ]:
# --- merge + eval (the merge is reclaimed in `finally` — ~17 GB) ----------------
CKPT = engine.arm_checkpoint(cfg).parent
merged = cfg.merged_dir / CKPT.name
MERGED_HERE = False
if merged.is_dir() and any(merged.iterdir()):
    print(f"OK    merged already present -> {merged}")
else:
    t0 = time.perf_counter()
    merged = engine.merge_checkpoint(cfg, CKPT)
    MERGED_HERE = True
    print(f"      merged in {time.perf_counter() - t0:.0f}s -> {merged}")

t0 = time.perf_counter()
try:
    cfg_eval = BaselineConfig(
        data_root=DATA_ROOT, model_path=merged, out_dir=RUN_DIR, run_name=RUN_TAG,
        max_pixels=1280 * 720, seed=42, n_eval=40 if SMOKE else None,
    )
    # The inference path must stay rung 06's EXACTLY. This rung's variable is the training recipe; a
    # post-processor or a second sample here would be a second variable and the delta would stop
    # being attributable.
    assert cfg_eval.answer_postprocess is None, "answer_postprocess must stay None"
    assert cfg_eval.n_samples == 1 and cfg_eval.enhance is None and cfg_eval.aux_view is None
    report = run_baseline(cfg_eval)
    print(f"eval done in {(time.perf_counter() - t0) / 60:.1f} min")
finally:
    if MERGED_HERE and not KEEP_MERGED and Path(merged).is_dir():
        shutil.rmtree(merged, ignore_errors=True)
        print(f"reclaimed ~17 GB -> removed {merged}")


In [ ]:
# --- canonical scoring + gates (all RAISE) --------------------------------------
# 🔴 Score ONLY via `frame.metrics` (RULES §1). Never re-derive a bucket, a floor or an accuracy
# inline — that is exactly how the same bug came back in rung 07.
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res = pd.read_csv(RUN_DIR / RUN_TAG / "results.csv")

missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
assert not missing, f"GATE 0 — {len(missing)} qIDs without gold; every margin would be inflated"

metrics.assert_no_dup_qid(res)
metrics.assert_ood_from_qid(res)
metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

# The mode gate, on the ARTIFACT rather than the variable: `-p SMOKE False` failing to take effect is
# otherwise indistinguishable from a successful full run.
assert (len(res) < 1000) == SMOKE, \
    f"MODE GATE FAILED: SMOKE={SMOKE} but the eval scored {len(res)} rows"
if not SMOKE:
    assert len(res) == 6252, f"expected the full eval set, got {len(res)} rows"

print("gates OK | bucket_mean:", round(strat["bucket_mean"], 4),
      "| margin_OOD:", round(strat["margin_OOD"], 4))


In [ ]:
# --- 🎯 THE DECLARED PRIMARY CELL: object_recognition_ID ------------------------
# Declared in PLAN.md §8a BEFORE this run existed. `bucket_mean`, `aggregation_ID`,
# `proxy_leaderboard` and `margin_OOD` are reported beside it as EXPLORATORY (RULES §S5) and are
# never quoted as the result.
def _cells(report: dict) -> dict:
    bb = pd.DataFrame(report["by_bucket"])
    out = {}
    for dist in ("ID", "OOD"):
        d = bb[bb.distribution == dist].set_index("capability_group")
        for name in ("aggregation", "object_recognition"):
            out[f"{name}_{dist}"] = float(d.loc[name, "accuracy"]) if name in d.index else float("nan")
    out["proxy_leaderboard"] = (out["aggregation_ID"] + out["object_recognition_ID"]) / 2
    out["bucket_mean"] = float(report["bucket_mean"])
    out["margin_OOD"] = float(report["margin_OOD"])
    return out

arm_c = _cells(strat)
if not SMOKE and any(pd.isna(v) for v in arm_c.values()):
    raise AssertionError(f"a scored bucket is missing from the full eval: {arm_c}")

# 🔴 The control is RECOMPUTED from A2 ep1's own archived answers through this same code path — a
# transcribed number cannot be re-derived, and this one can. PLAN.md §7 carries the transcription so
# that a mismatch is visible rather than silent.
ctrl_strat, ctrl_res = None, None
if (CTRL_DIR / "results.csv").exists():
    ctrl_res = pd.read_csv(CTRL_DIR / "results.csv")
    ctrl_strat = metrics.stratified_report(ctrl_res, gold=gold)
elif (CTRL_DIR / "stratified.json").exists():
    ctrl_strat = json.loads((CTRL_DIR / "stratified.json").read_text())

PREREG = {"object_recognition_ID": 0.6087962962962963, "aggregation_ID": 0.3884816753926701,
          "proxy_leaderboard": 0.4986389858444832, "bucket_mean": 0.5592175321379278,
          "margin_OOD": 0.16425}
if ctrl_strat is None:
    print(f"⚠️  no control artifacts under {CTRL_DIR} — deltas below are NOT computed")
    ctrl_c = {k: float("nan") for k in arm_c}
else:
    ctrl_c = _cells(ctrl_strat)
    drift = {k: (PREREG[k], ctrl_c[k]) for k in PREREG
             if k in ctrl_c and abs(ctrl_c[k] - PREREG[k]) > 1e-9}
    assert not drift, (f"the recomputed control does not match the PRE-REGISTERED numbers in "
                       f"PLAN.md §7: {drift} — the bar moved, which is the rung-38 failure")
    print("control reproduces PLAN.md §7 exactly")

print(pd.DataFrame([
    {"cell": k, "A2_ep1": round(ctrl_c[k], 4), "arm": round(arm_c[k], 4),
     "delta": round(arm_c[k] - ctrl_c[k], 4),
     "role": "🎯 PRIMARY" if k == "object_recognition_ID" else "exploratory"}
    for k in ("object_recognition_ID", "aggregation_ID", "object_recognition_OOD",
              "aggregation_OOD", "proxy_leaderboard", "bucket_mean", "margin_OOD")
]).to_string(index=False))


In [ ]:
# --- the PAIRED, VIDEO-CLUSTERED CI — the pre-registered decision input ----------
# Effective n is ~38 videos, not 6252 questions, so an unclustered CI would be ~10x too narrow and
# would manufacture significance. RULES §S8: only the declared cell may GRANT a win; ANY cell may
# veto — so every cell is computed, not just the primary one.
ci_df = None
if ctrl_res is not None:
    # RULES §2: leaf -> group ALWAYS via `Capability.group`. `metrics._leaf_to_group` is that single
    # canonical mapping (it raises on an un-mappable leaf); never a hand-kept dict, never a
    # group-name-vs-leaf filter, which silently dropped 964 rows in rung 07.
    a = ctrl_res[["qID", "video", "correctness", "primary"]].rename(
        columns={"correctness": "correct_a"})
    b = res[["qID", "correctness"]].rename(columns={"correctness": "correct_b"})
    j = a.merge(b, on="qID", how="inner")
    _need = len(res) if SMOKE else len(ctrl_res)
    assert len(j) == _need, f"arms not scored on the same questions ({len(j)} vs {_need})"
    j["group"] = j["primary"].map(metrics._leaf_to_group)
    j["dist"] = j["qID"].map(lambda q: "OOD" if str(q).split("__")[0] == "heico" else "ID")

    rows = []
    for grp in sorted(j["group"].unique()) + ["ALL"]:
        for dist in ("ID", "OOD"):
            sub = j[j["dist"] == dist]
            sub = sub if grp == "ALL" else sub[sub["group"] == grp]
            ci = metrics.paired_delta_ci(sub, n_boot=0 if SMOKE else 4000, seed=42)
            rows.append({"cell": f"{grp}_{dist}", **ci})
    ci_df = pd.DataFrame(rows)
    ci_df["excludes_zero"] = (ci_df.ci_low > 0) | (ci_df.ci_high < 0)
    ci_df["primary"] = ci_df.cell == "object_recognition_ID"
    print(ci_df.to_string(index=False))

    # 🔴 The pre-registered read, printed as a verdict and not left to the reader (PLAN.md §8b).
    if not SMOKE:
        prim = ci_df[ci_df.cell == "object_recognition_ID"].iloc[0]
        prim_ood = ci_df[ci_df.cell == "object_recognition_OOD"].iloc[0]
        granted = bool(prim.excludes_zero and prim.delta > 0)
        joint = bool(granted and prim_ood.delta > 0 and prim_ood.ci_low > 0)
        vetoed = ci_df[(ci_df.excludes_zero) & (ci_df.delta < 0)]["cell"].tolist()
        print(f"\nPRE-REGISTERED READ (vs {engine.A2_RUN} {engine.A2_ARM} {engine.A2_CHECKPOINT})")
        print(f"   (a) primary cell CI excludes 0 in the arm's favour : {granted} "
              f"({prim.delta:+.4f} [{prim.ci_low:+.4f}, {prim.ci_high:+.4f}])")
        print(f"   (b) holds on ID and OOD jointly                    : {joint}")
        print(f"   (c) no cell shows significant harm                 : {not vetoed} {vetoed}")
        print(f"   -> {'WIN' if (joint and not vetoed) else 'NOT A WIN'}")
        print("   A positive point estimate whose CI includes zero is a NULL (RULES §S8).")
        print("   Below |d| = 0.01 nothing is readable (§S4); acting needs |d| ~ 0.03 (§S1),")
        print("   and that is a team call with the cost stated, never automatic.")
else:
    print(f"⚠️  {CTRL_DIR}/results.csv absent — no paired CI, so there is no verdict to read")


In [ ]:
# --- class-balanced F1 on `fo_class` — MANDATORY before publishing (RULES §9b) ---
# `fo_class` accuracy is exact SET equality, so it is dominated by the head of a long-tailed class
# distribution and cannot see a tail collapse. Read the per_class table BESIDE the scalar: rung 21's
# +0.174 macro-F1 was 82% one `Needle` question flipping.
preds = metrics.predictions_frame(RUN_DIR / RUN_TAG)
f1 = metrics.class_f1_report(preds, gold, results_df=res, n_boot=0 if SMOKE else 2000)

ctrl_f1 = None
if (CTRL_DIR / "predictions.json").exists():
    ctrl_f1 = metrics.class_f1_report(metrics.predictions_frame(CTRL_DIR), gold, n_boot=0)

rows = []
for cell in ("pooled", "ID", "OOD"):
    b = f1[cell]
    a = ctrl_f1[cell] if ctrl_f1 else {"macro_f1": float("nan"), "exact_set_acc": float("nan")}
    rows.append({"cell": cell, "n": b["n"], "illegal": b["n_illegal"],
                 "macro_f1_A2": round(a["macro_f1"], 4), "macro_f1_arm": round(b["macro_f1"], 4),
                 "d_macro": round(b["macro_f1"] - a["macro_f1"], 4),
                 "exact_A2": round(a["exact_set_acc"], 4),
                 "exact_arm": round(b["exact_set_acc"], 4),
                 "ci_arm": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]"})
f1_df = pd.DataFrame(rows)
print(f1_df.to_string(index=False))

_per = pd.DataFrame(f1["ID"]["per_class"]).T
print("\n--- per class, ID (the tail is the point) ---")
if _per.empty:
    print("   (no fo_class x ID rows in this sample — expected in SMOKE only)")
else:
    print(_per[["n_gold", "recall", "precision", "f1"]]
          .sort_values("n_gold", ascending=False).round(3).to_string())


In [ ]:
# --- the run's row + persist (full runs only) -----------------------------------
row = {
    "run": RUN, "arm": "E_connector", "branch": gate_result["branch"], "epoch": 1,
    "checkpoint": CKPT.name, "baseline_run": engine.A2_RUN,
    "baseline_arm": engine.A2_ARM, "baseline_checkpoint": engine.A2_CHECKPOINT,
    "n_aligner": reached["n_aligner"], "n_llm": reached["n_llm"], "n_vit": reached["n_vit"],
    "freeze_aligner": cfg.freeze_aligner,
    "object_recognition_ID": arm_c["object_recognition_ID"],      # 🎯 the declared primary cell
    "aggregation_ID": arm_c["aggregation_ID"],
    "object_recognition_OOD": arm_c["object_recognition_OOD"],
    "aggregation_OOD": arm_c["aggregation_OOD"],
    "proxy_leaderboard": arm_c["proxy_leaderboard"],
    "bucket_mean": strat["bucket_mean"],
    "acc_ID": strat["acc_ID"], "acc_OOD": strat["acc_OOD"],
    "margin_ID": strat["margin_ID"], "margin_OOD": strat["margin_OOD"],
    # 🔴 the key names are `macro_f1_<dist>` because that is what
    # `metrics.assert_class_f1_reported` looks for (RULES §9b); renaming them disables the gate
    "macro_f1_ID": f1["ID"]["macro_f1"], "macro_f1_OOD": f1["OOD"]["macro_f1"],
    "d_primary_vs_A2_ep1": arm_c["object_recognition_ID"] - ctrl_c["object_recognition_ID"],
    "d_bucket_mean_vs_A2_ep1": arm_c["bucket_mean"] - ctrl_c["bucket_mean"],
}
print(pd.DataFrame([row]).T.to_string(header=False))

# RULES §9b — publishing without the class-balanced F1 RAISES.
metrics.assert_class_f1_reported(row, results_df=res)

if not SMOKE:
    # One CSV per arm, at the experiment root and OUTSIDE runs/ (which is gitignored, and where two
    # earlier pushes went to die). Read-concat-write is not atomic and two pods share this volume.
    out = EXP / "RESULTS.csv"
    df = pd.concat([pd.read_csv(out), pd.DataFrame([row])], ignore_index=True) if out.exists() \
        else pd.DataFrame([row])
    df.to_csv(out, index=False)
    f1_df.to_csv(EXP / f"RESULTS_class_f1_{RUN_TAG}.csv", index=False)
    if ci_df is not None:
        ci_df.to_csv(EXP / f"RESULTS_paired_ci_{RUN_TAG}.csv", index=False)
    ledger.register_run(RUN_DIR / RUN_TAG, strat, experiment="39-connector-lora",
                        run=f"{RUN}__{RUN_TAG}", model=f"39 E_connector epoch 1 ({CKPT.name})",
                        date="2026-08-13")
    print("wrote", out)
else:
    print("SMOKE — nothing persisted to RESULTS.csv or the ledger")


In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) -------
fo = res[res.answer_format == "fo_class"].merge(preds, on="qID")
print("--- fo_class misses (identity, not cardinality, is the usual failure) ---")
print(fo[fo.correctness == 0].head(12)[["qID", "prediction"]].to_string(index=False))

g = gold.copy()
g["template"] = g["question"].map(metrics.template_of)
clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
look = clips.merge(preds, on="qID").head(400)
if len(look):
    look["gold_n"] = look["answer"].map(metrics.read_count)
    look["pred_n"] = look["prediction"].map(metrics.read_count)
    print("\n--- predicted-vs-gold crosstab (Clips) ---")
    print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())


## After the run

1. **Read the declared primary cell first** — `object_recognition_ID` against A2 ep1's
   `0.6087962962962963`, with its paired video-clustered CI. Only that cell can grant a win; any
   cell can take one away (RULES §S8).
2. **A faithful negative is a real result.** If the connector received gradient (`n_aligner == 16`,
   `grad_norm > 0`) and the delta is null or negative, that is `PLAN.md` §9 shape 2 — record it in
   the ladder as NO-GO **with the number**, and do not re-cut the rung looking for a cell that
   clears the bar.
3. **Write the verdict** to `context/39-connector-lora/CONTEXT.md` §Results and, if it settles a
   question, to `context/decisions/` with frontmatter (`question`/`verdict`/`status`) — the
   `assert_decisions_indexed` gate RAISES on a note without it. Regenerate the ladder.
4. **Delete the rendered chain and the key file** from `/workspace/tmp` (CONSTITUTION §IX.1-2).
